# Notebook 10 — Orquestador end-to-end del pipeline (`predict.py`)

## Objetivo del notebook

Construir y validar el **orquestador end-to-end** que une los cuatro módulos
del sistema en un único pipeline. Hasta ahora cada módulo funcionaba de
forma independiente; este notebook diseña la función `procesar_informe`
que recibe un informe textual y devuelve un resultado consolidado.

## Arquitectura del pipeline

```
┌───────────────────────────────────────────────────────────────┐
│ Input: full_report (texto del informe mamográfico)            │
│                                                                │
│  Paso 1: extraer_birads(full_report)                          │
│              ↓ r_birads                                        │
│                                                                │
│  Paso 2: verificar_extraccion_birads(...)  ← TEMPRANO         │
│              ↓ verificacion_ml                                 │
│                                                                │
│  Paso 3: extraer_texto_recomendacion(...)                     │
│          clasificar_recomendacion(texto)                      │
│              ↓ r_recomendacion                                 │
│                                                                │
│  Paso 4: cotejar_birads_vs_recomendacion(                     │
│              r_birads, r_recomendacion,                        │
│              verificacion_ml=verificacion_ml)                 │
│              ↓ resultado_cotejo                                │
│                                                                │
│  Paso 5: Consolidar todo en un único dict de salida           │
└───────────────────────────────────────────────────────────────┘
```

## Decisiones de diseño

### 1. Orden del pipeline: verificación ML TEMPRANO

El verificador ML se ejecuta **inmediatamente después** de la extracción regex
del BI-RADS, no al final. Esto permite que:

- El cotejo (paso 4) reciba la confianza técnica ya ajustada
- Si la extracción es claramente errada, lo sabemos desde temprano
- La auditabilidad mejora: cada decisión usa toda la información disponible

### 2. Salida anidada por módulo

El resultado consolidado se organiza en bloques temáticos en lugar de un dict
plano:

```python
{
    "informe_id": ...,
    "timestamp": ...,
    "birads": {...},              # del módulo 1
    "verificacion_ml": {...},     # del módulo 4
    "recomendacion": {...},       # del módulo 2
    "cotejo_acr": {...},          # del módulo 3
    "confiabilidad_tecnica_global": ...,
}
```

### 3. Verificador ML opcional

Por defecto el verificador ML está activo (`usar_verificador_ml=True`). Se
puede desactivar para tests rápidos o casos sin disponibilidad de GPU/MPS.

### 4. Filosofía Human-on-the-Loop

Cada salida incluye trazabilidad completa: qué dijo cada módulo, qué regla
se aplicó, qué evidencia se consideró. El sistema **detecta y reporta**, no
decide clínicamente por el radiólogo.

## Sobre `predict.py` (modularización posterior)

Una vez validado el código en este notebook, se modulariza en
`src/predict.py` con:

- Función pública `procesar_informe`
- CLI con argparse: `python -m src.predict --input informe.txt`
- Tests inline
- Lazy loading del modelo ML (heredado del módulo 4)


---

## Paso 1 — Setup e imports


In [1]:
import sys
import os
import json
from datetime import datetime
from typing import Any, Dict, Optional

sys.path.insert(0, "..")

import pandas as pd
from tqdm import tqdm

from src.extractor_birads import extraer_birads
from src.extractor_recomendacion import (
    extraer_texto_recomendacion,
    clasificar_recomendacion,
)
from src.verificador_birads_ml import verificar_extraccion_birads
from src.cotejo_acr import cotejar_birads_vs_recomendacion

print("Imports OK")
print(f"  pandas: {pd.__version__}")


/Users/sebas/Documents/Proyectos_Doc/proyecto-ia-mamografia/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Imports OK
  pandas: 2.3.3


---

## Paso 2 — Función auxiliar `_construir_resultado_consolidado`

Toma los 4 resultados intermedios (uno por módulo) y los empaqueta en el
formato de salida final. Está separada para mantener la función pública
limpia y para facilitar tests del consolidador.


In [2]:
def _construir_resultado_consolidado(
    r_birads: Dict[str, Any],
    verificacion_ml: Optional[Dict[str, Any]],
    r_rec: Dict[str, Any],
    resultado_cotejo: Dict[str, Any],
    informe_id: Optional[str] = None,
) -> Dict[str, Any]:
    """Empaqueta los 4 resultados intermedios en el output final consolidado."""
    
    # Sub-dict: birads
    bloque_birads = {
        "valor": r_birads.get("birads_conclusion"),
        "confianza": r_birads.get("confianza"),
        "fuente": r_birads.get("fuente"),
        "encabezado_detectado": r_birads.get("encabezado_conclusion"),
        "menciones_adicionales": r_birads.get("menciones_adicionales", []),
    }
    
    # Sub-dict: verificacion_ml (puede ser None si se desactivó)
    if verificacion_ml is not None:
        bloque_verif = {
            "estado": verificacion_ml.get("estado_verificacion"),
            "birads_ml": verificacion_ml.get("birads_predicho_ml"),
            "confianza_ml": verificacion_ml.get("confianza_ml"),
            "coincide_con_regex": verificacion_ml.get("coincide_con_regex"),
            "regla_aplicada": verificacion_ml.get("regla_aplicada"),
            "mensaje": verificacion_ml.get("mensaje"),
        }
    else:
        bloque_verif = {"estado": "no_ejecutado", "birads_ml": None}
    
    # Sub-dict: recomendacion
    bloque_rec = {
        "texto_original": r_rec.get("trazabilidad", {}).get("texto_original", ""),
        "texto_normalizado": r_rec.get("trazabilidad", {}).get("texto_normalizado", ""),
        "categoria_principal": r_rec.get("categoria_principal"),
        "categorias_detectadas": r_rec.get("categorias_detectadas", []),
        "confianza": r_rec.get("confianza"),
        "metodo": r_rec.get("metodo"),
    }
    
    # Sub-dict: cotejo_acr
    bloque_cotejo = {
        "estado": resultado_cotejo.get("estado"),
        "alerta": resultado_cotejo.get("requiere_alerta"),
        "severidad": resultado_cotejo.get("severidad"),
        "recomendacion_esperada": resultado_cotejo.get("recomendacion_esperada"),
        "regla_aplicada": resultado_cotejo.get("trazabilidad", {}).get("regla_aplicada"),
        "mensaje": resultado_cotejo.get("mensaje"),
    }
    
    return {
        "informe_id": informe_id,
        "timestamp": datetime.now().isoformat(),
        "birads": bloque_birads,
        "verificacion_ml": bloque_verif,
        "recomendacion": bloque_rec,
        "cotejo_acr": bloque_cotejo,
        "confiabilidad_tecnica_global": resultado_cotejo.get("confiabilidad_tecnica"),
    }


print("Función _construir_resultado_consolidado lista.")


Función _construir_resultado_consolidado lista.


---

## Paso 3 — Función pública `procesar_informe`

Es el corazón del orquestador. Recibe un informe (texto completo y
opcionalmente el bloque de recomendaciones por separado) y devuelve el
resultado consolidado.

El flujo respeta el orden definido en el diseño:

1. **Extraer BI-RADS** (regex sobre la conclusión)
2. **Verificar con ML** (DistilBETO sobre la conclusión) — opcional
3. **Extraer y clasificar recomendación** (regex + TF-IDF)
4. **Cotejar** (BI-RADS vs recomendación, con verificación ML integrada)
5. **Consolidar** en el dict final


In [3]:
def procesar_informe(
    full_report: str,
    recommendations_col: Optional[str] = None,
    informe_id: Optional[str] = None,
    usar_verificador_ml: bool = True,
) -> Dict[str, Any]:
    """Procesa un informe mamográfico end-to-end.
    
    Args:
        full_report: texto completo del informe.
        recommendations_col: opcional, contenido de la columna Recommendations
            si está separada del full_report (más rápido que extraer del texto).
        informe_id: identificador opcional para auditoría.
        usar_verificador_ml: si False, omite la verificación ML (más rápido).
    
    Returns:
        Dict consolidado con la decisión clínica y trazabilidad completa,
        organizado por módulos: birads, verificacion_ml, recomendacion,
        cotejo_acr.
    """
    
    # ============================================================
    # PASO 1: Extraer BI-RADS
    # ============================================================
    r_birads = extraer_birads(full_report)
    
    # ============================================================
    # PASO 2: Verificar con ML (inmediato tras el regex)
    # ============================================================
    if usar_verificador_ml:
        verificacion_ml = verificar_extraccion_birads(
            full_report=full_report,
            birads_regex=r_birads.get("birads_conclusion"),
            confianza_regex=r_birads.get("confianza", "no_detectado"),
        )
    else:
        verificacion_ml = None
    
    # ============================================================
    # PASO 3: Extraer texto y clasificar recomendación
    # ============================================================
    texto_info = extraer_texto_recomendacion(
        recommendations_col=recommendations_col,
        full_report=full_report,
    )
    
    if texto_info["encontrado"]:
        r_rec = clasificar_recomendacion(
            texto_info["texto_normalizado"],
            es_ya_normalizado=True,
        )
        # Asegurar que la trazabilidad incluya el texto original
        r_rec["trazabilidad"]["texto_original"] = texto_info["texto"]
    else:
        # Sin bloque de recomendaciones → r_rec vacío (cotejo dirá no_procesable)
        r_rec = {
            "categorias_detectadas": [],
            "categoria_principal": None,
            "confianza": "no_clasificada",
            "metodo": None,
            "trazabilidad": {
                "texto_original": "",
                "texto_normalizado": "",
            },
        }
    
    # ============================================================
    # PASO 4: Cotejo BI-RADS vs recomendación (con verificación ML)
    # ============================================================
    resultado_cotejo = cotejar_birads_vs_recomendacion(
        resultado_birads=r_birads,
        resultado_recomendacion=r_rec,
        verificacion_ml=verificacion_ml,
    )
    
    # ============================================================
    # PASO 5: Consolidar todo en un único dict de salida
    # ============================================================
    return _construir_resultado_consolidado(
        r_birads=r_birads,
        verificacion_ml=verificacion_ml,
        r_rec=r_rec,
        resultado_cotejo=resultado_cotejo,
        informe_id=informe_id,
    )


print("Función procesar_informe lista.")


Función procesar_informe lista.


---

## Paso 4 — Tests sintéticos

Validamos el orquestador con 4 casos sintéticos que cubren los escenarios
principales:

1. **S1**: informe coherente claro (BI-RADS 2 + control anual)
2. **S2**: alerta crítica clara (BI-RADS 5 + control anual)
3. **S3**: alerta alta (BI-RADS 4 + criterio médico)
4. **S4**: sin verificador ML (modo rápido)


In [4]:
INFORME_COHERENTE = """
INFORME DE MAMOGRAFIA

HALLAZGOS: Mama densa heterogenea. Sin nodulos sospechosos.
Microcalcificaciones benignas dispersas.

CONCLUSION: BI-RADS 2 - Hallazgos benignos.

RECOMENDACIONES:
- Se sugiere control mamografico anual.
"""

INFORME_ALERTA_CRITICA = """
INFORME DE MAMOGRAFIA

HALLAZGOS: Masa irregular de 15mm con calcificaciones pleomorfas.
Altamente sospechosa de malignidad.

CONCLUSION: BI-RADS 5 - Lesion altamente sospechosa.

RECOMENDACIONES:
- Se sugiere control anual.
"""

INFORME_ALERTA_ALTA = """
INFORME DE MAMOGRAFIA

HALLAZGOS: Microcalcificaciones agrupadas en cuadrante superior externo.
Densidad sospechosa.

CONCLUSION: BI-RADS 4 - Hallazgo sospechoso.

RECOMENDACIONES:
- Controles segun criterio del medico tratante.
"""

print("=" * 75)
print("TESTS SINTÉTICOS DEL ORQUESTADOR")
print("=" * 75)

# Test 1: informe coherente
print("\n[S1] Informe coherente (BI-RADS 2 + control anual)")
r1 = procesar_informe(INFORME_COHERENTE, informe_id="test_S1")
print(f"  birads.valor: {r1['birads']['valor']}")
print(f"  cotejo.estado: {r1['cotejo_acr']['estado']}")
print(f"  cotejo.alerta: {r1['cotejo_acr']['alerta']}")
print(f"  verif_ml.estado: {r1['verificacion_ml']['estado']}")
assert r1['birads']['valor'] == 2, "BI-RADS debe ser 2"
assert r1['cotejo_acr']['alerta'] == False, "No debe haber alerta"
print("  PASA")

# Test 2: alerta crítica
print("\n[S2] Alerta crítica (BI-RADS 5 + control anual)")
r2 = procesar_informe(INFORME_ALERTA_CRITICA, informe_id="test_S2")
print(f"  birads.valor: {r2['birads']['valor']}")
print(f"  cotejo.estado: {r2['cotejo_acr']['estado']}")
print(f"  cotejo.alerta: {r2['cotejo_acr']['alerta']}")
print(f"  cotejo.severidad: {r2['cotejo_acr']['severidad']}")
assert r2['birads']['valor'] == 5, "BI-RADS debe ser 5"
assert r2['cotejo_acr']['alerta'] == True, "Debe haber alerta"
assert r2['cotejo_acr']['severidad'] == "critica", "Severidad debe ser critica"
print("  PASA")

# Test 3: alerta alta
print("\n[S3] Alerta alta (BI-RADS 4 + criterio medico)")
r3 = procesar_informe(INFORME_ALERTA_ALTA, informe_id="test_S3")
print(f"  birads.valor: {r3['birads']['valor']}")
print(f"  cotejo.estado: {r3['cotejo_acr']['estado']}")
print(f"  cotejo.severidad: {r3['cotejo_acr']['severidad']}")
assert r3['birads']['valor'] == 4, "BI-RADS debe ser 4"
assert r3['cotejo_acr']['alerta'] == True, "Debe haber alerta"
assert r3['cotejo_acr']['severidad'] == "alta", "Severidad debe ser alta"
print("  PASA")

# Test 4: sin verificador ML
print("\n[S4] Sin verificador ML (modo rapido)")
r4 = procesar_informe(INFORME_COHERENTE, informe_id="test_S4", usar_verificador_ml=False)
print(f"  verif_ml.estado: {r4['verificacion_ml']['estado']}")
print(f"  cotejo.estado: {r4['cotejo_acr']['estado']}")
assert r4['verificacion_ml']['estado'] == "no_ejecutado", "ML debe estar desactivado"
assert r4['cotejo_acr']['alerta'] == False, "Sin alerta"
print("  PASA")

print("\n4/4 tests sintéticos pasados.")


TESTS SINTÉTICOS DEL ORQUESTADOR

[S1] Informe coherente (BI-RADS 2 + control anual)
  birads.valor: 2
  cotejo.estado: coherente
  cotejo.alerta: False
  verif_ml.estado: confirmado
  PASA

[S2] Alerta crítica (BI-RADS 5 + control anual)
  birads.valor: 5
  cotejo.estado: incoherente
  cotejo.alerta: True
  cotejo.severidad: critica
  PASA

[S3] Alerta alta (BI-RADS 4 + criterio medico)
  birads.valor: 4
  cotejo.estado: incoherente
  cotejo.severidad: alta
  PASA

[S4] Sin verificador ML (modo rapido)
  verif_ml.estado: no_ejecutado
  cotejo.estado: coherente
  PASA

4/4 tests sintéticos pasados.


---

## Paso 5 — Test sobre un informe real del corpus

Tomamos uno de los informes reales del corpus y ejecutamos el orquestador.
Inspeccionamos el output completo para verificar que todos los campos vengan
poblados correctamente.


In [5]:
# Cargar el corpus
df_corpus = pd.read_csv("../data/processed/reports_cleaned.csv")
print(f"Corpus cargado: {len(df_corpus)} informes")

# Tomar un informe BI-RADS 4 (para ver una alerta real)
df_birads4 = df_corpus[df_corpus["BI-RADS"] == 4].head(5)
print(f"\nInformes BI-RADS 4 disponibles: {len(df_birads4)}")

# Procesar el primero
idx_prueba = df_birads4.index[0]
row = df_corpus.iloc[idx_prueba]
print(f"\nProcesando informe idx={idx_prueba}...\n")

resultado = procesar_informe(
    full_report=row["Full_Report"],
    recommendations_col=row.get("Recommendations"),
    informe_id=f"informe_{idx_prueba:04d}",
)

# Mostrar el resultado de forma legible
print(json.dumps(resultado, indent=2, ensure_ascii=False, default=str)[:2500])


Corpus cargado: 4357 informes

Informes BI-RADS 4 disponibles: 5

Procesando informe idx=112...

{
  "informe_id": "informe_0112",
  "timestamp": "2026-06-20T00:38:26.924976",
  "birads": {
    "valor": 4,
    "confianza": "alta",
    "fuente": "bloque_conclusion_estricto",
    "encabezado_detectado": "CONCLUSIÓN",
    "menciones_adicionales": []
  },
  "verificacion_ml": {
    "estado": "ml_no_confirma",
    "birads_ml": 0,
    "confianza_ml": 0.5954,
    "coincide_con_regex": false,
    "regla_aplicada": "regla_3_regex_alta_ml_disiente",
    "mensaje": "Regex (confianza alta) extrajo BI-RADS 4. ML predijo BI-RADS 0 con confianza 0.60. Se prioriza la extracción regex (literal del informe). El ML puede haber confundido un patrón estadístico. Sin alerta clínica."
  },
  "recomendacion": {
    "texto_original": "- SE SUGIERE ECOGRAFÍA MAMARIA Y CARACTERIZACIÓN HISTOLÓGICA.",
    "texto_normalizado": "- se sugiere ecografia mamaria y caracterizacion histologica.",
    "categoria_principal

---

## Paso 6 — Aplicación a una muestra de 100 informes

Procesamos 100 informes del corpus para validar que el orquestador funciona
sobre datos reales sin errores, y medimos el rendimiento aproximado.


In [9]:
import time

# Muestrear 100 informes representativos PRESERVANDO el índice original del corpus
muestra = df_corpus.sample(n=100, random_state=42)  # SIN reset_index

print(f"Procesando {len(muestra)} informes con verificador ML activo...")
t0 = time.time()

resultados_muestra = []
for idx_original, row in tqdm(muestra.iterrows(), total=len(muestra)):
    # idx_original es el índice del corpus, no de la muestra
    resultado = procesar_informe(
        full_report=row["Full_Report"],
        recommendations_col=row.get("Recommendations"),
        informe_id=f"informe_{idx_original:04d}",  # ← usa el índice original
    )
    resultados_muestra.append(resultado)

tiempo_total = time.time() - t0
tiempo_por_informe = tiempo_total / len(muestra)

print(f"\nTiempo total: {tiempo_total:.1f}s")
print(f"Tiempo por informe: {tiempo_por_informe*1000:.0f}ms")
print(f"Procesados: {len(resultados_muestra)} informes")
print(f"Indices del corpus en la muestra (primeros 5): {list(muestra.index[:5])}")

Procesando 100 informes con verificador ML activo...


100%|██████████| 100/100 [00:00<00:00, 162.11it/s]


Tiempo total: 0.6s
Tiempo por informe: 6ms
Procesados: 100 informes
Indices del corpus en la muestra (primeros 5): [1721, 1393, 1609, 1309, 2772]


---

## Paso 7 — Análisis del output consolidado

Verificamos que todos los campos del output vengan poblados, que la
distribución de estados sea razonable y que el sistema no produzca
resultados inesperados.


In [10]:
from collections import Counter

print("=" * 75)
print("DISTRIBUCION DE RESULTADOS EN LA MUESTRA")
print("=" * 75)

# Distribución por BI-RADS extraído
birads_dist = Counter(r["birads"]["valor"] for r in resultados_muestra)
print("\nBI-RADS extraidos:")
for k in sorted(birads_dist.keys(), key=lambda x: (x is None, x)):
    print(f"  {k}: {birads_dist[k]}")

# Distribución por estado de cotejo
estado_dist = Counter(r["cotejo_acr"]["estado"] for r in resultados_muestra)
print("\nEstados de cotejo:")
for k, v in sorted(estado_dist.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v}")

# Distribución por estado de verificación ML
verif_dist = Counter(r["verificacion_ml"]["estado"] for r in resultados_muestra)
print("\nEstados de verificacion ML:")
for k, v in sorted(verif_dist.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v}")

# Distribución por confiabilidad técnica
conf_dist = Counter(r["confiabilidad_tecnica_global"] for r in resultados_muestra)
print("\nConfiabilidad tecnica global:")
for k, v in sorted(conf_dist.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v}")

# Alertas detectadas
alertas = [r for r in resultados_muestra if r["cotejo_acr"]["alerta"]]
print(f"\nAlertas clinicas detectadas: {len(alertas)} / {len(resultados_muestra)} ({100*len(alertas)/len(resultados_muestra):.1f}%)")

if alertas:
    print("\nPrimera alerta detectada:")
    a = alertas[0]
    print(f"  informe_id: {a['informe_id']}")
    print(f"  birads: {a['birads']['valor']}")
    print(f"  estado: {a['cotejo_acr']['estado']}")
    print(f"  severidad: {a['cotejo_acr']['severidad']}")
    print(f"  recomendacion_detectada: {a['recomendacion']['categoria_principal']}")
    print(f"  recomendacion_esperada: {a['cotejo_acr']['recomendacion_esperada']}")
    print(f"  mensaje: {a['cotejo_acr']['mensaje']}")


DISTRIBUCION DE RESULTADOS EN LA MUESTRA

BI-RADS extraidos:
  0: 26
  1: 10
  2: 58
  3: 1
  4: 3
  5: 2

Estados de cotejo:
  coherente: 66
  coherente_equivalente: 25
  coherente_con_precaucion: 5
  incoherente: 4

Estados de verificacion ML:
  confirmado: 98
  ml_no_confirma: 1
  discrepante_real: 1

Confiabilidad tecnica global:
  alta: 99
  baja: 1

Alertas clinicas detectadas: 4 / 100 (4.0%)

Primera alerta detectada:
  informe_id: informe_0838
  birads: 0
  estado: incoherente
  severidad: alta
  recomendacion_detectada: criterio_medico
  recomendacion_esperada: estudio_complementario_imagen
  mensaje: BI-RADS 0 indica que el estudio está incompleto y requiere completarse con imágenes adicionales o búsqueda de exámenes previos. La recomendación detectada no apunta a resolver el estudio incompleto a corto plazo.


---

## Paso 8 — Comparación con cotejo v2 del notebook 09

Verificamos que el orquestador produce los **mismos estados clínicos** que
el cotejo v2 ejecutado en el notebook 09. Esto confirma que el orquestador
no introduce inconsistencias.


In [12]:
# Cargar los resultados del cotejo v2
df_cotejo_v2 = pd.read_csv("./anexos/resultados_cotejo_v2.csv")
print(f"Cotejo v2 cargado: {len(df_cotejo_v2)} filas")

# Comparar la muestra contra el cotejo v2
coincidencias = 0
diferencias = []

for r in resultados_muestra:
    informe_id = r["informe_id"]
    idx = int(informe_id.split("_")[1])
    
    # Buscar en el cotejo v2
    fila_v2 = df_cotejo_v2.iloc[idx]
    
    estado_predict = r["cotejo_acr"]["estado"]
    estado_v2 = fila_v2["estado"]
    
    if estado_predict == estado_v2:
        coincidencias += 1
    else:
        diferencias.append({
            "informe_id": informe_id,
            "estado_predict": estado_predict,
            "estado_v2": estado_v2,
            "birads_predict": r["birads"]["valor"],
            "birads_v2": fila_v2["birads"],
        })

print(f"\nCoincidencias: {coincidencias}/{len(resultados_muestra)}")
print(f"Diferencias: {len(diferencias)}")

if diferencias:
    print("\nPrimeras 3 diferencias:")
    for d in diferencias[:3]:
        print(f"  {d}")
else:
    print("\nOK: el orquestador produce los mismos estados que el cotejo v2.")


Cotejo v2 cargado: 4357 filas

Coincidencias: 100/100
Diferencias: 0

OK: el orquestador produce los mismos estados que el cotejo v2.


---

## Paso 9 — Guardar resultados de la muestra

Persistimos los 100 resultados procesados como ejemplo de uso del
orquestador. Esto sirve como anexo del notebook y como dataset de prueba
para validaciones futuras.


In [13]:
# Guardar el JSON con todos los resultados
ruta_json = "./anexos/predict_muestra_100.json"
with open(ruta_json, "w", encoding="utf-8") as f:
    json.dump(resultados_muestra, f, indent=2, ensure_ascii=False, default=str)
print(f"OK Guardado: {ruta_json}")

# También un CSV resumido para inspección rápida
filas_csv = []
for r in resultados_muestra:
    filas_csv.append({
        "informe_id": r["informe_id"],
        "birads": r["birads"]["valor"],
        "birads_confianza": r["birads"]["confianza"],
        "verificacion_ml": r["verificacion_ml"]["estado"],
        "birads_ml": r["verificacion_ml"]["birads_ml"],
        "recomendacion": r["recomendacion"]["categoria_principal"],
        "estado_cotejo": r["cotejo_acr"]["estado"],
        "alerta": r["cotejo_acr"]["alerta"],
        "severidad": r["cotejo_acr"]["severidad"],
        "confiabilidad_tecnica": r["confiabilidad_tecnica_global"],
    })

df_csv = pd.DataFrame(filas_csv)
ruta_csv = "./anexos/predict_muestra_100.csv"
df_csv.to_csv(ruta_csv, index=False)
print(f"OK Guardado: {ruta_csv}")
print(f"  Filas: {len(df_csv)}")
print(f"  Columnas: {list(df_csv.columns)}")


OK Guardado: ./anexos/predict_muestra_100.json
OK Guardado: ./anexos/predict_muestra_100.csv
  Filas: 100
  Columnas: ['informe_id', 'birads', 'birads_confianza', 'verificacion_ml', 'birads_ml', 'recomendacion', 'estado_cotejo', 'alerta', 'severidad', 'confiabilidad_tecnica']


---

## Conclusiones

### Lo logrado

1. **Orquestador end-to-end funcionando**: la función `procesar_informe`
   ejecuta los 4 módulos en el orden correcto y produce salida consolidada.

2. **Verificación ML inmediata**: el módulo 4 se ejecuta justo después del
   módulo 1, no al final, respetando el principio de auditabilidad temprana.

3. **Tests sintéticos pasados (4/4)**: el orquestador maneja correctamente
   informes coherentes, alertas críticas, alertas altas y modo sin ML.

4. **Consistencia con cotejo v2**: los estados clínicos producidos por el
   orquestador coinciden con los del notebook 09 (no introduce variaciones).

5. **Salida estructurada por módulo**: el dict consolidado organiza la
   información por bloques (birads, verificacion_ml, recomendacion,
   cotejo_acr) facilitando consumo por dashboards o sistemas downstream.

### Decisiones validadas

- **Orden temporal correcto**: ML inmediato tras el regex (no al final)
- **Verificador ML opcional**: parámetro `usar_verificador_ml` para tests rápidos
- **Lazy loading**: el modelo DistilBETO se carga solo cuando se necesita
- **Trazabilidad completa**: cada bloque del output incluye campos de auditoría

### Limitaciones reconocidas

1. **Procesamiento secuencial por informe**: no hay batching de inferencias ML.
   Para producción podría agregarse procesamiento por lotes.
2. **Sin manejo de errores explícito**: si un módulo lanza excepción, propaga.
   En `predict.py` modularizado se podría agregar manejo con try/except.
3. **Test sobre 100 informes**: validación más amplia (4 357 completos)
   queda para una sesión específica si es necesario.

### Siguiente paso

**Modularizar a `src/predict.py`** con:

- Función pública `procesar_informe` (idéntica a esta)
- Función auxiliar `_construir_resultado_consolidado` (privada)
- **CLI con argparse**: `python -m src.predict --input informe.txt`
- Tests inline ejecutables con `python -m src.predict`
- Imports limpios desde los 4 módulos
